In [1]:
import datetime as dt
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.io as pio
import plotly.graph_objects as go
import scipy.stats as stats
from plotly.subplots import make_subplots
import plotly.offline as pyo
pyo.init_notebook_mode(connected = True)
pio.renderers.default = 'iframe'
pd.options.plotting.backend = 'plotly'
import pylab


In [2]:
end = dt.datetime.now()
start = dt.datetime(2021,1,1)

df = yf.download(["AAPL","MSFT", "TSLA", "NVDA", "AMD"], start, end)
# df.columns = df.columns.droplevel(1)
df.head()

[*********************100%***********************]  5 of 5 completed


Price            Close                                                \
Ticker            AAPL        AMD        MSFT       NVDA        TSLA   
Date                                                                   
2021-01-04  125.856712  92.300003  208.406525  13.076020  243.256668   
2021-01-05  127.412766  92.769997  208.607574  13.366437  245.036667   
2021-01-06  123.123840  90.330002  203.198532  12.578445  251.993332   
2021-01-07  127.325233  95.160004  208.980957  13.305859  272.013336   
2021-01-08  128.424225  94.580002  210.254242  13.238801  293.339996   

Price             High                                                ...  \
Ticker            AAPL        AMD        MSFT       NVDA        TSLA  ...   
Date                                                                  ...   
2021-01-04  129.941387  96.059998  213.490075  13.613480  248.163330  ...   
2021-01-05  128.122732  93.209999  209.201144  13.405076  246.946671  ...   
2021-01-06  127.451658  92.279999  207.257721  13.207143  258.000000  ...   
2021-01-07  128.015745  95.510002  209.986182  13.339513  272.329987  ...   
2021-01-08  128.988301  96.400002  211.173309  13.382639  294.829987  ...   

Price             Open                                                \
Ticker            AAPL        AMD        MSFT       NVDA        TSLA   
Date                                                                   
2021-01-04  129.853862  92.110001  213.040117  13.066797  239.820007   
2021-01-05  125.350981  92.099998  207.994868  13.062308  241.220001   
2021-01-06  124.213090  91.620003  203.121942  13.184707  252.830002   
2021-01-07  124.835528  91.330002  204.912199  12.930435  259.209991   
2021-01-08  128.793781  95.980003  209.354326  13.324306  285.333344   

Price          Volume                                            
Ticker           AAPL       AMD      MSFT       NVDA       TSLA  
Date                                                             
2021-01-04  143301900  51802600  37130100  560640000  145914600  
2021-01-05   97664900  34208000  23823000  322760000   96735600  
2021-01-06  155088000  51911700  35930700  580424000  134100000  
2021-01-07  109578200  42897200  27694500  461480000  154496700  
2021-01-08  105158200  39816400  22956200  292528000  225166500  

[5 rows x 25 columns]

In [3]:
log_returns = np.log(df.Close/df.Close.shift(1)).dropna()
log_returns

Ticker,AAPL,AMD,MSFT,NVDA,TSLA
Date,,,,,
2021-01-05,0.012288,0.005079,0.000964,0.021967,0.007291
2021-01-06,-0.034241,-0.026654,-0.026271,-0.060762,0.027995
2021-01-07,0.033554,0.052090,0.028060,0.056220,0.076448
2021-01-08,0.008594,-0.006114,0.006074,-0.005052,0.075481
2021-01-11,-0.023523,0.027839,-0.009746,0.025635,-0.081442
...,...,...,...,...,...
2026-05-01,0.031880,0.016923,0.016200,-0.005628,0.023796
2026-05-04,-0.011886,-0.054138,-0.001981,0.000151,0.004315
2026-05-05,0.026204,0.039385,-0.005430,-0.010026,-0.008032


In [4]:
daily_std = log_returns.std()
daily_std

Ticker
AAPL    0.017381
AMD     0.033743
MSFT    0.016525
NVDA    0.032108
TSLA    0.037325
dtype: float64

In [5]:
annualized_vol = daily_std*np.sqrt(252)
annualized_vol*100

Ticker
AAPL    27.592233
AMD     53.565111
MSFT    26.232390
NVDA    50.970271
TSLA    59.252251
dtype: float64

In [6]:
fig = make_subplots(rows=2, cols=2)

trace0 = go.Histogram(x=log_returns['AAPL'], name='AAPL')
trace1 = go.Histogram(x=log_returns['MSFT'], name='Microsoft')
trace2 = go.Histogram(x=log_returns['TSLA'], name='Tesla')
trace3 = go.Histogram(x=log_returns['NVDA'], name='NVIDIA')
trace4 = go.Histogram(x=log_returns['AMD'], name='AMD')

fig.append_trace(trace0, 1, 1)
fig.append_trace(trace1, 1, 2)
fig.append_trace(trace2, 2, 1)
fig.append_trace(trace3, 2, 2)
# trace4 dropped — no 5th slot in a 2x2 grid

fig.update_layout(autosize = False)

fig.show()

In [8]:
TRADING_DAYS = 60
volatility = log_returns.rolling(window = TRADING_DAYS).std()*np.sqrt(TRADING_DAYS)

In [9]:
volatility.plot()

In [14]:
rf = 0.01/252
sharpe_ratio = (log_returns.rolling(window=TRADING_DAYS).mean()-rf)*TRADING_DAYS/volatility

In [15]:
sharpe_ratio.plot()

In [16]:
sortino_vol = log_returns[log_returns<0].rolling(window = TRADING_DAYS, center=True, min_periods = 10).std()*np.sqrt(TRADING_DAYS)

In [17]:
sortino_ratio = (log_returns.rolling(window=TRADING_DAYS).mean()-rf)*TRADING_DAYS/sortino_vol

In [18]:
sortino_vol.plot()

In [19]:
sortino_ratio.plot()

In [22]:
m2_ratio = pd.DataFrame()
benchmark_vol = volatility['AAPL']
for c in log_returns.columns:
    if c != 'AAPL' :
        m2_ratio[c] = (sharpe_ratio[c]*benchmark_vol/TRADING_DAYS + rf)*TRADING_DAYS

In [23]:
m2_ratio.plot()

In [25]:
def max_drawdown(retunrs):
    cumulative_returns = (1+returns).cumprod()
    peak = cumulative_returns.expanding(min_periods = 1).max()
    drawdown = (cumulative_returns/peak) - 1
    return drawdown.min()

returns = df.Close.pct_change().dropna()
max_drawdowns = returns.apply(max_drawdown, axis = 0)
max_drawdowns*100

Ticker,AAPL,AMD,MSFT,NVDA,TSLA
Ticker,,,,,
AAPL,-33.360519,-33.360519,-33.360519,-33.360519,-33.360519
AMD,-65.449943,-65.449943,-65.449943,-65.449943,-65.449943
MSFT,-37.148476,-37.148476,-37.148476,-37.148476,-37.148476
NVDA,-66.335092,-66.335092,-66.335092,-66.335092,-66.335092
TSLA,-73.632217,-73.632217,-73.632217,-73.632217,-73.632217


In [26]:
calmars = np.exp(log_returns.mean()*252)/abs(max_drawdowns)
calmars.plot.bar()